In [1]:
%load_ext autoreload
%autoreload 2

### Install SDG
```bash 
pip install sdg-hub
pip install rich datasets tabulate transformers
```
 - If you haven't already, run the document pre-processing notebook to create the seed data

In [2]:
# Third Party
from datasets import load_dataset
from openai import OpenAI

# First Party
from sdg_hub import Flow
from sdg_hub import FlowMetadata, FlowParameter
from sdg_hub.core.flow import FlowRegistry

/workspace/home/lab/.conda/envs/sdg_hub/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[19:52:52] INFO     HTTP Request: GET                                                               ]8;id=201282;file:///workspace/home/lab/.conda/envs/sdg_hub/lib/python3.11/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=288911;file:///workspace/home/lab/.conda/envs/sdg_hub/lib/python3.11/site-packages/httpx/_client.py#1025\1025]8;;\
                    https://raw.githubusercontent.com/BerriAI/litellm/main/model_prices_and_context                
                    _window.json "HTTP/1.1 200 OK"                                                                 

In [3]:
# Required to run the flow with async mode
import nest_asyncio

nest_asyncio.apply()  

### Run SDG
- This will create knowledge flow from provided yaml file
- We will run this on small dataset for demo purposes
- For large scale generation, please use the python command provided in the next cell
- You can analyze the generated data to ensure the quality is similar to proivded QnA pairs

In [4]:
# Auto-discover all available flows (no setup needed!)
FlowRegistry.discover_flows()

# List available flows
flows = FlowRegistry.list_flows()
print(f"Available flows: {flows}")

# You can also search the flows by tag
qa_flows = FlowRegistry.search_flows(tag="question-generation")
print(f"QA flows: {qa_flows}")

[19:52:53] INFO     Discovered 1 flows                                                              ]8;id=679481;file:///workspace/home/lab/abhi/sdg_hub_pr_178/src/sdg_hub/core/flow/registry.py\registry.py]8;;\:]8;id=313508;file:///workspace/home/lab/abhi/sdg_hub_pr_178/src/sdg_hub/core/flow/registry.py#110\110]8;;\

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━┓
┃ Name                                                                            ┃ Author   ┃ Tags     ┃ Descri… ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━┩
│ Advanced Document Grounded Question-Answer Generation Flow for Knowledge Tuning │ SDG Hub  │ questio… │ A       │
│                                                                                 │ Contrib… │ knowled… │ compre… │
│                                                                                 │          │ qa-pair… │ flow    │
│                                                                                 │          │ documen… │ that    │
│                                                                                 │          │ educati… │ genera… │
│                                                                                 │          │          │ high-q… │
│                                                                                 │          │          │ questi… │
│                                                                                 │          │          │ pairs   │
│                                                                                 │          │          │ from    │
│                                                                                 │          │          │ input   │
│                                                                                 │          │          │ docume… │
│                                                                                 │          │          │ using   │
│                                                                                 │          │          │ multip… │
│                                                                                 │          │          │ LLM     │
│                                                                                 │          │          │ blocks  │
│                                                                                 │          │          │ for     │
│                                                                                 │          │          │ questi… │
│                                                                                 │          │          │ genera… │
│                                                                                 │          │          │ answer  │
│                                                                                 │          │          │ synthe… │
│                                                                                 │          │          │ and     │
│                                                                                 │          │          │ quality │
│                                                                                 │          │          │ evalua… │
└─────────────────────────────────────────────────────────────────────────────────┴──────────┴──────────┴─────────┘

Available flows: ['Advanced Document Grounded Question-Answer Generation Flow for Knowledge Tuning']
QA flows: ['Advanced Document Grounded Question-Answer Generation Flow for Knowledge Tuning']


In [5]:
# We will use the "Advanced Document Grounded Question-Answer Generation Flow for Knowledge Tuning" flow.
# For loading the flow simply use the fullname to load it
flow_name = "Advanced Document Grounded Question-Answer Generation Flow for Knowledge Tuning"
flow_path = FlowRegistry.get_flow_path(flow_name)
flow = Flow.from_yaml(flow_path)

           INFO     Loading flow from:                                                                  ]8;id=729870;file:///workspace/home/lab/abhi/sdg_hub_pr_178/src/sdg_hub/core/flow/base.py\base.py]8;;\:]8;id=965241;file:///workspace/home/lab/abhi/sdg_hub_pr_178/src/sdg_hub/core/flow/base.py#140\140]8;;\
                    /workspace/home/lab/abhi/sdg_hub_pr_178/src/sdg_hub/flows/qa_generation/document_gr            
                    ounded_qa/multi_summary_qa/instructlab/flow.yaml                                               

In [6]:
# Configure the flow to use a vllm model hosted at localhost:8000/v1. 
# You can dynamically change the model without having to change the flow yaml file.
flow.get_default_model()

flow.get_model_recommendations()

flow.set_model_config(
    model="hosted_vllm/meta-llama/Llama-3.3-70B-Instruct",
    api_base="http://localhost:8000/v1",
    api_key="EMPTY",
)

           INFO     Auto-detected 7 LLM blocks for configuration: ['eval_faithfulness',                 ]8;id=110114;file:///workspace/home/lab/abhi/sdg_hub_pr_178/src/sdg_hub/core/flow/base.py\base.py]8;;\:]8;id=375297;file:///workspace/home/lab/abhi/sdg_hub_pr_178/src/sdg_hub/core/flow/base.py#547\547]8;;\
                    'eval_relevancy', 'gen_atomic_facts', 'gen_detailed_summary',                                  
                    'gen_extractive_summary', 'knowledge_generation', 'verify_question']                           

           INFO     Loaded LLM client for model                                                ]8;id=581317;file:///workspace/home/lab/abhi/sdg_hub_pr_178/src/sdg_hub/core/blocks/llm/client_manager.py\client_manager.py]8;;\:]8;id=570252;file:///workspace/home/lab/abhi/sdg_hub_pr_178/src/sdg_hub/core/blocks/llm/client_manager.py#61\61]8;;\
                    'hosted_vllm/meta-llama/Llama-3.3-70B-Instruct'                                                

           INFO     Initialized LLMChatBlock 'gen_detailed_summary' with model                ]8;id=210886;file:///workspace/home/lab/abhi/sdg_hub_pr_178/src/sdg_hub/core/blocks/llm/llm_chat_block.py\llm_chat_block.py]8;;\:]8;id=492243;file:///workspace/home/lab/abhi/sdg_hub_pr_178/src/sdg_hub/core/blocks/llm/llm_chat_block.py#264\264]8;;\
                    'hosted_vllm/meta-llama/Llama-3.3-70B-Instruct'                                                

           INFO     Loaded LLM client for model                                                ]8;id=529705;file:///workspace/home/lab/abhi/sdg_hub_pr_178/src/sdg_hub/core/blocks/llm/client_manager.py\client_manager.py]8;;\:]8;id=551366;file:///workspace/home/lab/abhi/sdg_hub_pr_178/src/sdg_hub/core/blocks/llm/client_manager.py#61\61]8;;\
                    'hosted_vllm/meta-llama/Llama-3.3-70B-Instruct'                                                

           INFO     Initialized LLMChatBlock 'gen_atomic_facts' with model                    ]8;id=17282;file:///workspace/home/lab/abhi/sdg_hub_pr_178/src/sdg_hub/core/blocks/llm/llm_chat_block.py\llm_chat_block.py]8;;\:]8;id=298479;file:///workspace/home/lab/abhi/sdg_hub_pr_178/src/sdg_hub/core/blocks/llm/llm_chat_block.py#264\264]8;;\
                    'hosted_vllm/meta-llama/Llama-3.3-70B-Instruct'                                                

...

           INFO     Initialized LLMChatBlock 'eval_relevancy_llm_chat' with model             ]8;id=51841;file:///workspace/home/lab/abhi/sdg_hub_pr_178/src/sdg_hub/core/blocks/llm/llm_chat_block.py\llm_chat_block.py]8;;\:]8;id=162224;file:///workspace/home/lab/abhi/sdg_hub_pr_178/src/sdg_hub/core/blocks/llm/llm_chat_block.py#264\264]8;;\
                    'hosted_vllm/meta-llama/Llama-3.3-70B-Instruct'                                                

           INFO     Loaded LLM client for model                                                ]8;id=835767;file:///workspace/home/lab/abhi/sdg_hub_pr_178/src/sdg_hub/core/blocks/llm/client_manager.py\client_manager.py]8;;\:]8;id=104880;file:///workspace/home/lab/abhi/sdg_hub_pr_178/src/sdg_hub/core/blocks/llm/client_manager.py#61\61]8;;\
                    'hosted_vllm/meta-llama/Llama-3.3-70B-Instruct'                                                

           INFO     Initialized LLMChatBlock 'verify_question_llm_chat' with model            ]8;id=688098;file:///workspace/home/lab/abhi/sdg_hub_pr_178/src/sdg_hub/core/blocks/llm/llm_chat_block.py\llm_chat_block.py]8;;\:]8;id=644505;file:///workspace/home/lab/abhi/sdg_hub_pr_178/src/sdg_hub/core/blocks/llm/llm_chat_block.py#264\264]8;;\
                    'hosted_vllm/meta-llama/Llama-3.3-70B-Instruct'                                                

           INFO     Successfully configured 7 LLM blocks with: model:                                   ]8;id=241904;file:///workspace/home/lab/abhi/sdg_hub_pr_178/src/sdg_hub/core/flow/base.py\base.py]8;;\:]8;id=845385;file:///workspace/home/lab/abhi/sdg_hub_pr_178/src/sdg_hub/core/flow/base.py#586\586]8;;\
                    'hosted_vllm/meta-llama/Llama-3.3-70B-Instruct', api_base:                                     
                    'http://localhost:8000/v1', api_key: EMPTY                                                     

           INFO     Configured blocks: ['eval_faithfulness', 'eval_relevancy', 'gen_atomic_facts',      ]8;id=306600;file:///workspace/home/lab/abhi/sdg_hub_pr_178/src/sdg_hub/core/flow/base.py\base.py]8;;\:]8;id=472503;file:///workspace/home/lab/abhi/sdg_hub_pr_178/src/sdg_hub/core/flow/base.py#589\589]8;;\
                    'gen_detailed_summary', 'gen_extractive_summary', 'knowledge_generation',                      
                    'verify_question']                                                                             

In [7]:
# Load the seed data
number_of_samples = 2
seed_data_dir = f"sdg_demo_output/"
ds = load_dataset('json', data_files=f'{seed_data_dir}/seed_data.jsonl', split='train')
ds = ds.shuffle(seed=42).select(range(number_of_samples))

In [8]:
# Generate data
generated_data = flow.generate(ds)

[19:52:59] INFO     Starting flow 'Advanced Document Grounded Question-Answer Generation Flow for       ]8;id=386627;file:///workspace/home/lab/abhi/sdg_hub_pr_178/src/sdg_hub/core/flow/base.py\base.py]8;;\:]8;id=743923;file:///workspace/home/lab/abhi/sdg_hub_pr_178/src/sdg_hub/core/flow/base.py#379\379]8;;\
                    Knowledge Tuning' v1.0.0 with 2 samples across 18 blocks                                       

           INFO     Executing block 1/18: duplicate_document_col (DuplicateColumnsBlock)                ]8;id=322677;file:///workspace/home/lab/abhi/sdg_hub_pr_178/src/sdg_hub/core/flow/base.py\base.py]8;;\:]8;id=542924;file:///workspace/home/lab/abhi/sdg_hub_pr_178/src/sdg_hub/core/flow/base.py#393\393]8;;\

╭──────────────────────────────────────────── duplicate_document_col ─────────────────────────────────────────────╮
│ 📊 Processing Input Data                                                                                        │
│ Block Type: DuplicateColumnsBlock                                                                               │
│ Input Rows: 2                                                                                                   │
│ Input Columns: 11                                                                                               │
│ Column Names: document_outline, document_title, domain, icl_document, icl_query_1, icl_response_1, icl_query_2, │
│ icl_response_2, icl_query_3, icl_response_3, document                                                           │
│ Expected Output Columns: base_document                                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────── duplicate_document_col - Complete ───────────────────────────────────────╮
│ ✅ Processing Complete                                                                                          │
│ Rows: 2 → 2                                                                                                     │
│ Columns: 11 → 12                                                                                                │
│ 🟢 Added: base_document                                                                                         │
│ 📋 Final Columns: base_document, document, document_outline, document_title, domain, icl_document, icl_query_1, │
│ icl_query_2, icl_query_3, icl_response_1, icl_response_2, icl_response_3                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

           INFO     Block 'duplicate_document_col' completed successfully: 2 samples, 12 columns        ]8;id=130226;file:///workspace/home/lab/abhi/sdg_hub_pr_178/src/sdg_hub/core/flow/base.py\base.py]8;;\:]8;id=732553;file:///workspace/home/lab/abhi/sdg_hub_pr_178/src/sdg_hub/core/flow/base.py#425\425]8;;\

...

Filter: 100%|██████████| 39/39 [00:00<00:00, 4522.22 examples/s]


           INFO     Question verification completed: 39 → 9 samples (filtered 30       ]8;id=444439;file:///workspace/home/lab/abhi/sdg_hub_pr_178/src/sdg_hub/core/blocks/evaluation/verify_question_block.py\verify_question_block.py]8;;\:]8;id=893207;file:///workspace/home/lab/abhi/sdg_hub_pr_178/src/sdg_hub/core/blocks/evaluation/verify_question_block.py#429\429]8;;\
                    samples)                                                                                       

╭────────────────────────────────────────── verify_question - Complete ───────────────────────────────────────────╮
│ ✅ Processing Complete                                                                                          │
│ Rows: 39 → 9                                                                                                    │
│ Columns: 31 → 35                                                                                                │
│ 🟢 Added: raw_verify_question, verification_explanation, verification_rating, verify_question_prompt            │
│ 📋 Final Columns: atomic_facts_prompt, dataset_type, document, document_outline, document_title, domain,        │
│ eval_faithfulness_prompt, eval_relevancy_prompt, extractive_summary_prompt, faithfulness_explanation,           │
│ faithfulness_judgment, icl_document, icl_query_1, icl_query_2, icl_query_3, icl_response_1, icl_response_2,     │
│ icl_response_3, knowledge_generation_prompt, question, raw_atomic_facts, raw_document, raw_eval_faithfulness,   │
│ raw_eval_relevancy, raw_knowledge_generation, raw_summary_detailed, raw_summary_extractive,                     │
│ raw_verify_question, relevancy_explanation, relevancy_score, response, summary_prompt,                          │
│ verification_explanation, verification_rating, verify_question_prompt                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

           INFO     Block 'verify_question' completed successfully: 9 samples, 35 columns               ]8;id=260980;file:///workspace/home/lab/abhi/sdg_hub_pr_178/src/sdg_hub/core/flow/base.py\base.py]8;;\:]8;id=237695;file:///workspace/home/lab/abhi/sdg_hub_pr_178/src/sdg_hub/core/flow/base.py#425\425]8;;\

           INFO     Flow 'Advanced Document Grounded Question-Answer Generation Flow for Knowledge      ]8;id=870695;file:///workspace/home/lab/abhi/sdg_hub_pr_178/src/sdg_hub/core/flow/base.py\base.py]8;;\:]8;id=228888;file:///workspace/home/lab/abhi/sdg_hub_pr_178/src/sdg_hub/core/flow/base.py#439\439]8;;\
                    Tuning' completed successfully: 9 final samples, 35 final columns                              

### Converting the generated data into training format

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
from knowledge_utils import create_knowledge_regular_ds, create_knowledge_pretraining_ds

from datasets import concatenate_datasets

output_dir = f"sdg_demo_output/"


# Create Pretraining Knowledge Dataset (Also known as Phase 0.7/Phase 7)
instructlab_phase_1_ds = create_knowledge_pretraining_ds(generated_data)
instructlab_phase_1_ds.to_json(f'{output_dir}/instructlab_phase_1_ds.jsonl', orient='records', lines=True)

# Create Regular Knowledge Dataset (Also known as Phase 1.0/Phase 10)
instructlab_phase_2_ds = create_knowledge_regular_ds(generated_data)

# Mix the pre-computed skills with the regular knowledge dataset. If more than one dataset were generated simply add those in this concatenation stage.
# If you have any generated instruction data, that can be also mixed in this stage. If you only have generated skills phase 07 generation and training can be skipped.
instructlab_phase_2_ds.to_json(f'{output_dir}/instructlab_phase_2_ds.jsonl', orient='records', lines=True)

In [ ]:
# If you have any other instruction tuning datasets you can mix with phase 2 dataset.
instruction_tuning_dataset_path = "<Your instruction tuning dataset path>"
instruction_tuning_dataset = load_dataset('json', data_files=instruction_tuning_dataset_path, split='train')
instructlab_phase_2_ds = concatenate_datasets([instructlab_phase_2_ds, instruction_tuning_dataset])
instructlab_phase_2_ds.to_json(f'{output_dir}/instructlab_phase_2_ds.jsonl', orient='records', lines=True)